## Multi-Category HTML Parsing

### Overview
Extends the AC-only parser (notebook 01) to handle **3 appliance categories**: AC, Refrigerator, Washing Machine.
Goal: produce a unified `appliances_parsed.csv` with a `category` column for downstream multi-category modeling.

### Workflow
1. Inspect the structure of `refrigerator.html` and `washing_machines.html` to verify selectors work
2. Print a sample card from each category to discover per-category feature structure
3. Write per-category parser logic (cells 6-7)
4. Concatenate all 3 categories and save the unified CSV

### Why 3 categories, not 4
Microwave is **out of scope** for this pass. Re-evaluate after notebook 10 (multi-category model experiments).

In [1]:
import pandas as pd
import numpy as np
import re
from bs4 import BeautifulSoup
import sys
sys.path.append("D:/Study/data_science/underpriced-listing-predictor/scraper")

import ac_selectors as s

In [2]:
html_files = {
    'AC':            'D:/Study/data_science/underpriced-listing-predictor/data/01.raw/ac_full_dump.html',
    'Refrigerator':  'D:/Study/data_science/underpriced-listing-predictor/data/01.raw/refrigerator.html',
    'WashingMachine':'D:/Study/data_science/underpriced-listing-predictor/data/01.raw/washing_machines.html',
}
# NOTE: Microwave is out of scope for this pass. Add later if needed.

for label, path in html_files.items():
    with open(path, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f.read(), 'lxml')
    cards = soup.find_all('div', {'class':'sm-product has-tag has-features has-actions'})
    print(f"{label}: {len(cards)} cards")

AC: 1020 cards
Refrigerator: 1020 cards
WashingMachine: 885 cards


In [3]:
for label, path in html_files.items():
    with open(path, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f.read(), 'lxml')
    cards = soup.find_all('div', {'class':'sm-product has-tag has-features has-actions'})
    if not cards:
        print(f"\n=== {label}: NO CARDS FOUND ===")
        # Fallback: see what class strings actually exist
        all_sm = soup.find_all('div', {'class': re.compile(r'sm-product')})
        if all_sm:
            print(f"  Found {len(all_sm)} divs matching 'sm-product'. Sample class string:")
            print(f"  '{all_sm[0].get('class')}'")
        continue
    sample = cards[0]
    print(f"\n=== {label} sample card ===")
    print(f"Name: {sample.find('h2').text}")
    print(f"Price: {sample.find('span', {'class':'price'}).text}")
    features = sample.find('ul', {'class':'sm-feat specs'}).find_all('li')
    print(f"Features ({len(features)}):")
    for i, f in enumerate(features):
        print(f"  [{i}] {f.text}")


=== AC sample card ===
Name: Whirlpool SAI18B52MCD1 1.5 Ton 5 Star Inverter Split AC
Price: ₹24,990
Features (8):
  [0] Split AC
  [1] 1.5 Ton Capacity
  [2] Air Swing, Auto Restart
  [3] Turbo Mode, Sleep Mode
  [4] Dust Filter
  [5] Self Diagnosis
  [6] Night Glow Buttons
  [7] 5 Star Rating

=== Refrigerator sample card ===
Name: Samsung RT28C3733S8 236 L 3 Star Double Door Refrigerator
Price: ₹25,490
Features (8):
  [0] Multi Door
  [1] Top Mounted Freezer
  [2] 236 L Capacity
  [3] 3 Star Energy Rating
  [4] Toughened Glass, Moisture Control
  [5] Convertible, Frost Free
  [6] Deodorizer
  [7] Door Alarm

=== WashingMachine sample card ===
Name: Samsung Ecobubble WA80BG4545BY 8 kg Fully Automatic Top Load Washing Machine
Price: ₹13,294
Features (8):
  [0] Washing Machine with Dryer
  [1] Fully Automatic, Top Load
  [2] 8 kg Capacity
  [3] PP Dual Wing Pulsator
  [4] Red LED Display
  [5] Inverter Technology
  [6] Timer, Self Clean
  [7] Child Lock


### Per-category parser design (fill this in after running cell 4)

From the sample output, fill in the actual feature positions:

| Category      | feature[0]    | feature[1]     | feature[2:8] (sub-features) |
|---------------|---------------|----------------|----------------------------|
| AC            | ac_type       | capacity (ton) | Inverter, Wi-Fi, ... |
| Refrigerator  | ?             | ?              | ? |
| WashingMachine| ?             | ?              | ? |

**Column naming decision (generic vs specific)**:
- Generic: `cat_feature_1`, `capacity_raw` → keeps parser simple, unit normalization happens in 08
- Specific: `ac_type`, `door_type`, `load_type` → more readable but creates NaN columns

Plan defaults to **generic**. Override here if you prefer specific.

In [13]:
def parse_category(html_path, label):
    with open(html_path, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f.read(), 'lxml')
    
    cards = soup.find_all('div', {'class': 'sm-product has-tag has-features has-actions'})
    
    rows = []
    for card in cards:
        # Structured fields (clean extraction — no position dependency)
        name    = card.find('h2').text.strip()
        price   = card.find('span', {'class': 'price'}).text.strip()
        try:
            rating = card.find('div', {'class': 'rating'}) \
                        .find('span', {'class': 'sm-rating'})['style']
        except:
            rating = np.nan
        
        # Aggregate: flatten ALL features (split comma-separated li items)
        features_raw = card.find('ul', {'class': 'sm-feat specs'}).find_all('li')
        features_flat = []
        for li in features_raw:
            for item in li.text.strip().split(','):
                item = item.strip()
                if item:
                    features_flat.append(item)
        
        rows.append({
            'name':     name,
            'price':    price,
            'rating':   rating,
            'category': label,
            'features': features_flat,
        })
    
    return pd.DataFrame(rows)


In [15]:
# Run all 3 parsers, concatenate, save
dfs = []
for label, path in html_files.items():
    df = parse_category(path, label)
    dfs.append(df)

all_appliances = pd.concat(dfs, ignore_index=True)
print(f"Total: {len(all_appliances)} rows")
print(all_appliances['category'].value_counts())

all_appliances.to_csv(
    'D:/Study/data_science/underpriced-listing-predictor/data/02.parsed/all_appliances_parsed.csv',
    index=False
)

Total: 2925 rows
category
AC                1020
Refrigerator      1020
WashingMachine     885
Name: count, dtype: int64


### Next steps

Once `appliances_parsed.csv` is saved, the pipeline continues:

- **08.ac_multi_category_cleaning.ipynb** — unit normalization (tons→kW? liters→L, kg→kg), missing values, brand extraction, per-category feature engineering
- **09.ac_multi_category_eda.ipynb** — combined EDA with `category` as a feature, check for category-conditional price distributions
- **10.ac_multi_category_model_experiments.ipynb** — re-run 9-model comparison on combined data (target: R² ≥ 0.70)
- **11.ac_multi_category_shap.ipynb** — SHAP on the multi-category winner
- **12.ac_streamlit_app.ipynb** — UI with category selector